In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp, chi2_contingency, wasserstein_distance, entropy
from itertools import combinations


# 1. CSV-Dateien einlesen (Pfad zu realen und synthetischen Daten anpassen)
real_path = "../../data/data_for_CTGAN/CTGAN_basedata.csv"
syn_path = "../../data/synthetic_result_data/536_synthetic_train_data.csv"

df_real = pd.read_csv(real_path)
df_syn = pd.read_csv(syn_path)


In [ ]:
def compute_subscores(original: pd.DataFrame, synthetic: pd.DataFrame, alpha_kl=1.0, mmd_gamma=1.0, weights=None):
    """
    Compute Statistical Similarity Score (SSS) subscores for each feature and overall score.
    Returns a dict with per-feature subscores and the overall SSS.
    """
    def compute_mmd(x, y, gamma):
        """Compute unbiased MMD^2 with RBF kernel."""
        X = x.values.reshape(-1, 1)
        Y = y.values.reshape(-1, 1)
        pairwise = lambda A, B: np.exp(-gamma * (A[:, None] - B[None, :])**2)
        Kxx = pairwise(X, X)
        Kyy = pairwise(Y, Y)
        Kxy = pairwise(X, Y)
        n, m = len(X), len(Y)
        mmd2 = (np.sum(Kxx) - np.trace(Kxx)) / (n*(n-1)) \
             + (np.sum(Kyy) - np.trace(Kyy)) / (m*(m-1)) \
             - 2 * np.sum(Kxy) / (n*m)
        return np.sqrt(max(mmd2, 0))
    
    if weights is None:
        # Equal weights for 9 metrics (some may be skipped per feature)
        weights = {i: 1/9 for i in range(1, 10)}
    
    results = {}
    feature_scores = []
    
    for col in original.columns:
        o = original[col]
        s = synthetic[col]
        subscores = {}
        
        # Determine feature type
        if pd.api.types.is_numeric_dtype(o):
            # 1. Mean difference
            R = o.max() - o.min()
            delta_mean = abs(o.mean() - s.mean())
            subscores['mean'] = max(0, 1 - delta_mean / R) if R > 0 else 1
            
            # 2. Median difference
            delta_median = abs(o.median() - s.median())
            subscores['median'] = max(0, 1 - delta_median / R) if R > 0 else 1
            
            # 3. Variance difference
            V = max(o.var(), s.var())
            delta_var = abs(o.var() - s.var())
            subscores['variance'] = max(0, 1 - delta_var / V) if V > 0 else 1
            
            # 4. KS test
            p_ks = ks_2samp(o, s).pvalue
            subscores['ks'] = p_ks
            
            # 6. Wasserstein distance
            w_dist = wasserstein_distance(o, s)
            subscores['wasserstein'] = 1 / (1 + w_dist)
            
            # 7. MMD
            mmd_val = compute_mmd(o, s, mmd_gamma)
            subscores['mmd'] = 1 / (1 + mmd_val)
            
            # 8. KL divergence (on histograms)
            bins = min(50, max(len(o.unique()), len(s.unique())))
            p_hist, _ = np.histogram(o, bins=bins, density=True)
            q_hist, _ = np.histogram(s, bins=bins, density=True)
            p_hist += 1e-8  # Laplace smoothing
            q_hist += 1e-8
            kl_div = entropy(p_hist, q_hist)
            subscores['kl'] = np.exp(-alpha_kl * kl_div)
            
            # 9. Coverage = proportion of original's range covered
            cov = (len(np.intersect1d(o.unique(), s.unique())) / len(o.unique())) if len(o.unique()) > 0 else 1
            subscores['coverage'] = cov
            
        else:
            # Categorical
            # 5. Chi-squared test
            ct = pd.crosstab(o, s)
            chi2, p_chi, *_ = chi2_contingency(ct, correction=False)
            subscores['chi2'] = p_chi
            # 9. Coverage
            cov = (len(set(o.unique()) & set(s.unique())) / len(o.unique())) if len(o.unique()) > 0 else 1
            subscores['coverage'] = cov

        # Normalize weights if some metrics skipped
        valid_metrics = subscores.keys()
        total_w = sum(weights[idx] for idx, name in enumerate(
            ['mean','median','variance','ks','chi2','wasserstein','mmd','kl','coverage'], start=1) if name in valid_metrics)
        
        # Compute feature score
        score = sum(weights[idx] * subscores[name] for idx, name in enumerate(
            ['mean','median','variance','ks','chi2','wasserstein','mmd','kl','coverage'], start=1) if name in subscores)
        score /= total_w
        
        results[col] = {
            'subscores': subscores,
            'feature_score': score
        }
        feature_scores.append(score)
    
    # Overall SSS
    overall_sss = np.mean(feature_scores)
    return {'per_feature': results, 'SSS': overall_sss}

In [ ]:

result = compute_subscores(df_real, df_syn)
print("Statistical Similarity Score (SSS):", result['SSS'])
for feature, data in result['per_feature'].items():
    print(f"\nFeature: {feature}")
    print(" Subscores:", data['subscores'])
    print(" Feature score:", data['feature_score'])

